# UGSC-ML: Zero-Shot Multilingual Sentiment Analysis Pipeline

**Paper:** *Cross-Lingual Sentiment Classification in Sustainable Mobility: A Zero-Shot Domain Transfer Evaluation Framework*  
**Authors:** Ainhoa Serna, Jon Kepa Gerrikagoitia, Juan de Oña  
**Journal:** AI (MDPI), 2026  
**Dataset & Code:** https://doi.org/10.5281/zenodo.15085522  
**GitHub:** https://github.com/ainhoaserna/UGSC-multilingual-sentiment

---

## What this notebook does

This notebook reproduces the complete experimental pipeline from the paper:

1. **Setup** — installs dependencies and downloads the model from Hugging Face  
2. **Load dataset** — reads the UGSC-ML multilingual dataset (375 sentences × 5 languages)  
3. **Run inference** — applies XLM-RoBERTa in zero-shot domain transfer mode (Algorithm 1)  
4. **Analyse results** — confidence distributions, language-level patterns  
5. **Generate figures** — reproduces Figures 1, 2 and 3 from the paper  

**Runtime:** ~5 minutes on GPU (T4), ~15 minutes on CPU  
**No local installation required** — runs entirely in the browser.

> 💡 **Tip:** Go to `Runtime → Change runtime type → T4 GPU` for faster inference.

---
## 1. Setup

In [ ]:
# Install required packages
!pip install -q transformers torch scipy tqdm matplotlib

In [ ]:
import os
import csv
import unicodedata
from collections import Counter

import torch
import torch.nn.functional as F
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from tqdm.notebook import tqdm
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# Device selection
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {DEVICE}')
if DEVICE == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

---
## 2. Load Dataset

The UGSC-ML dataset is available on Zenodo: https://doi.org/10.5281/zenodo.15085522

**Option A** — Upload from your computer (click the folder icon on the left → upload)  
**Option B** — Download automatically from Zenodo (run the cell below)

In [ ]:
# Option B: Download dataset from Zenodo automatically
ZENODO_DOI = '10.5281/zenodo.20200844'
DATASET_FILE = 'ugsc_ml_translations.csv'

if not os.path.exists(DATASET_FILE):
    print('Downloading dataset from Zenodo...')
    # Zenodo API to get download URL
    import urllib.request, json
    record_id = ZENODO_DOI.split('.')[-1]
    api_url = f'https://zenodo.org/api/records/{record_id}'
    with urllib.request.urlopen(api_url) as r:
        record = json.loads(r.read())
    files = {f['key']: f['links']['self'] for f in record['files']}
    if DATASET_FILE in files:
        urllib.request.urlretrieve(files[DATASET_FILE], DATASET_FILE)
        print(f'Downloaded: {DATASET_FILE}')
    else:
        print(f'Available files: {list(files.keys())}')
        print('Please download the dataset manually from https://doi.org/10.5281/zenodo.15085522')
else:
    print(f'Dataset already present: {DATASET_FILE}')

In [ ]:
# Load dataset
LANGUAGES = ['English', 'Spanish', 'French', 'German', 'Italian']

df_wide = pd.read_csv(DATASET_FILE, encoding='utf-8-sig')
print(f'Loaded: {len(df_wide)} sentences × {len(LANGUAGES)} languages = {len(df_wide)*len(LANGUAGES)} predictions')
print(f'Columns: {list(df_wide.columns)}')
df_wide.head(3)

---
## 3. Load Model

**Model:** `cardiffnlp/twitter-xlm-roberta-base-sentiment`  
Applied in **zero-shot domain transfer** mode: pre-trained on CC100 + fine-tuned on Twitter sentiment, applied without task- or domain-specific adaptation to transport reviews (Section 2.2).

In [ ]:
MODEL_NAME = 'cardiffnlp/twitter-xlm-roberta-base-sentiment'

print(f'Loading model: {MODEL_NAME}')
print('(First run downloads ~1 GB — cached for subsequent runs)\n')

tokeniser = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)
model = model.to(DEVICE)
model.eval()

# Label mapping
id2label = model.config.id2label
LABEL_MAP = {v.lower(): v.lower() for v in id2label.values()}
print(f'Model labels: {id2label}')
print(f'Model ready on {DEVICE}')

---
## 4. Run Inference — Algorithm 1

Implements **Algorithm 1** from Section 2.4 of the paper:

```
For each sentence s:
  1. Preprocess s (Unicode normalisation, strip whitespace)
  2. Tokenise using SentencePiece tokeniser
  3. Run inference → logits
  4. Apply softmax → confidence distribution
  5. Predicted class = argmax
  6. Confidence = max softmax score
  7. If confidence < 0.5 → flag for qualitative taxonomy analysis
```

In [ ]:
# Step 1: Preprocessing (Section 2.4)
def preprocess(text: str) -> str:
    """Minimal preprocessing: Unicode NFKC normalisation + strip."""
    text = unicodedata.normalize('NFKC', text)
    text = ' '.join(text.strip().split())
    return text

# Steps 2-6: Batch inference
def predict_batch(texts, batch_size=32, max_length=512, threshold=0.5):
    """Run inference on a list of texts. Returns list of result dicts."""
    results = []
    batches = [texts[i:i+batch_size] for i in range(0, len(texts), batch_size)]

    for batch in tqdm(batches, desc='Classifying'):
        encoded = tokeniser(
            batch,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors='pt'
        )
        encoded = {k: v.to(DEVICE) for k, v in encoded.items()}

        with torch.no_grad():
            logits = model(**encoded).logits

        probs = F.softmax(logits, dim=-1).cpu().numpy()

        for prob_row in probs:
            best_idx = int(prob_row.argmax())
            sentiment = id2label[best_idx].lower()
            confidence = float(prob_row[best_idx])
            results.append({
                'Sentiment':          sentiment,
                'Confidence':         round(confidence, 10),
                'low_confidence_flag': confidence < threshold,
            })
    return results

In [ ]:
# Build long-format record list
records = []
for _, row in df_wide.iterrows():
    sid = str(row.get('sentence_id', '')).zfill(4)
    en_source = str(row.get('English', ''))
    for lang in LANGUAGES:
        text = str(row.get(lang, ''))
        if text.strip():
            records.append({
                'sentence_id':    sid,
                'Language':       lang,
                'Text':           text,
                'English_source': en_source,
            })

print(f'Total records: {len(records)}')

# Run inference
texts = [preprocess(r['Text']) for r in records]
predictions = predict_batch(texts, batch_size=32, threshold=0.5)

# Combine into DataFrame
for rec, pred in zip(records, predictions):
    rec.update(pred)

df_results = pd.DataFrame(records)
print(f'\nDone. Shape: {df_results.shape}')
df_results.head()

In [ ]:
# Save results
OUTPUT_FILE = 'sentiment_classification_results.csv'
df_results.to_csv(OUTPUT_FILE, index=False, encoding='utf-8')
print(f'Results saved to: {OUTPUT_FILE}')

# Summary statistics (Table 1)
print('\n── Results summary (Table 1) ──')
for lang in LANGUAGES:
    ld = df_results[df_results['Language'] == lang]
    dist = ld['Sentiment'].value_counts()
    lc = ld['low_confidence_flag'].sum()
    print(f"  {lang:<10} "
          f"neg={dist.get('negative',0):3d}  "
          f"neu={dist.get('neutral',0):3d}  "
          f"pos={dist.get('positive',0):3d}  "
          f"| low-conf={lc} ({lc/len(ld)*100:.1f}%)")

total_lc = df_results['low_confidence_flag'].sum()
print(f'\n  Total low-confidence: {total_lc}/{len(df_results)} ({total_lc/len(df_results)*100:.1f}%)')

---
## 5. Generate Figures

Reproduces **Figures 1, 2 and 3** from the paper.

In [ ]:
# Colour palette
C_NEG = '#C0392B'
C_NEU = '#7F8C8D'
C_POS = '#2980B9'

os.makedirs('figures', exist_ok=True)

# ── Figure 1: Donut chart ──────────────────────────────────────
overall = df_results['Sentiment'].value_counts()
neg, neu, pos = overall.get('negative',0), overall.get('neutral',0), overall.get('positive',0)
grand = neg + neu + pos
sizes  = [neg, neu, pos]
colors = [C_NEG, C_NEU, C_POS]
labels = ['Negative', 'Neutral', 'Positive']
pcts   = [n/grand*100 for n in sizes]

fig, ax = plt.subplots(figsize=(6.5, 4.8))
fig.patch.set_facecolor('white')
ax.pie(sizes, colors=colors, startangle=90,
       wedgeprops=dict(width=0.52, edgecolor='white', linewidth=2.5),
       counterclock=False)
ax.text(0,  0.07, f'n = {grand:,}', ha='center', va='center', fontsize=11, color='#555555')
ax.text(0, -0.18, 'predictions',   ha='center', va='center', fontsize=9,  color='#888888')
patches = [mpatches.Patch(facecolor=c, edgecolor='white') for c in colors]
legend_labels = [f'{l}  {n:,}  ({p:.1f}%)' for l,n,p in zip(labels,sizes,pcts)]
ax.legend(patches, legend_labels, loc='lower center', bbox_to_anchor=(0.5,-0.12),
          ncol=3, frameon=False, fontsize=10)
ax.set_aspect('equal')
plt.tight_layout()
plt.savefig('figures/figure1_donut.png', dpi=180, bbox_inches='tight', facecolor='white')
plt.show()
print('Figure 1 saved.')

In [ ]:
# ── Figure 2: Stacked bar ─────────────────────────────────────
n_per_lang = len(df_results) // len(LANGUAGES)
neg_counts, neu_counts, pos_counts = [], [], []
for lang in LANGUAGES:
    ld = df_results[df_results['Language']==lang]['Sentiment'].value_counts()
    neg_counts.append(ld.get('negative',0))
    neu_counts.append(ld.get('neutral', 0))
    pos_counts.append(ld.get('positive',0))

neg_pct = [n/n_per_lang*100 for n in neg_counts]
neu_pct = [n/n_per_lang*100 for n in neu_counts]
pos_pct = [n/n_per_lang*100 for n in pos_counts]

fig, ax = plt.subplots(figsize=(8, 4.2))
fig.patch.set_facecolor('white')
ax.barh(LANGUAGES, neg_pct, color=C_NEG, label='Negative', height=0.55)
ax.barh(LANGUAGES, neu_pct, left=neg_pct, color=C_NEU, label='Neutral', height=0.55)
ax.barh(LANGUAGES, pos_pct,
        left=[neg_pct[i]+neu_pct[i] for i in range(len(LANGUAGES))],
        color=C_POS, label='Positive', height=0.55)

for i in range(len(LANGUAGES)):
    ax.text(neg_pct[i]/2, i, f'{neg_counts[i]}\n({neg_pct[i]:.0f}%)',
            ha='center', va='center', fontsize=8, color='white', fontweight='bold')
    if neu_pct[i] >= 8:
        ax.text(neg_pct[i]+neu_pct[i]/2, i, f'{neu_counts[i]}\n({neu_pct[i]:.0f}%)',
                ha='center', va='center', fontsize=8, color='white', fontweight='bold')
    else:
        ax.text(neg_pct[i]+neu_pct[i]+1.2, i, f'{neu_counts[i]} ({neu_pct[i]:.0f}%)',
                ha='left', va='center', fontsize=7.5, color='#444444')
    ax.text(neg_pct[i]+neu_pct[i]+pos_pct[i]/2, i, f'{pos_counts[i]}\n({pos_pct[i]:.0f}%)',
            ha='center', va='center', fontsize=8, color='white', fontweight='bold')

ax.set_xlabel('Percentage of predictions (%)', fontsize=10)
ax.set_xlim(0,100)
ax.set_xticks([0,25,50,75,100])
ax.xaxis.grid(True, linestyle='--', alpha=0.4, color='#cccccc')
ax.set_axisbelow(True)
for sp in ['top','right','left']:
    ax.spines[sp].set_visible(False)
ax.tick_params(axis='y', length=0, labelsize=10)
ax.tick_params(axis='x', labelsize=9)
ax.legend(loc='lower right', frameon=True, fontsize=9, framealpha=0.9,
          edgecolor='#cccccc', ncol=3)
plt.tight_layout()
plt.savefig('figures/figure2_stacked_bar.png', dpi=180, bbox_inches='tight', facecolor='white')
plt.show()
print('Figure 2 saved.')

In [ ]:
# ── Figure 3: Combined bar + dot ──────────────────────────────
# Requires low_confidence_annotated_113.csv
# Download from Zenodo or upload manually

LC_FILE = 'low_confidence_annotated_113.csv'

CATEGORY_ORDER = [
    'Irony / Sarcasm', 'Informal Punctuation',
    'Conditional / Hypothetical', 'Idiomatic Expressions',
    'Mixed Sentiment', 'Translation Drift',
]
CATEGORY_LABELS = [
    'Irony /\nSarcasm', 'Informal\nPunctuation',
    'Conditional /\nHypothetical', 'Idiomatic\nExpressions',
    'Mixed\nSentiment', 'Translation\nDrift',
]
CAT_COLORS = [
    '#8E44AD','#566573','#1A5276','#117A65','#B7950B','#922B21'
]

if os.path.exists(LC_FILE):
    df_lc = pd.read_csv(LC_FILE, encoding='utf-8')
    n_total = len(df_lc)
    print(f'Loaded {n_total} annotated low-confidence cases')

    n_cases, pct_vals, conf_vals = [], [], []
    for cat in CATEGORY_ORDER:
        cr = df_lc[df_lc['final_category']==cat]
        n = len(cr)
        pct = n/n_total*100
        conf = pd.to_numeric(cr['Confidence'], errors='coerce').mean()
        n_cases.append(n)
        pct_vals.append(pct)
        conf_vals.append(conf)

    weighted_mean = sum(c*n for c,n in zip(conf_vals,n_cases))/sum(n_cases)

    fig, (ax1,ax2) = plt.subplots(1,2,figsize=(10,4.2),
                                   gridspec_kw={'width_ratios':[2,1]})
    fig.patch.set_facecolor('white')

    ax1.barh(CATEGORY_LABELS, pct_vals, color=CAT_COLORS, height=0.55)
    for i,(p,n) in enumerate(zip(pct_vals,n_cases)):
        ax1.text(p+0.3,i,f'{p:.1f}%  (n={n})',va='center',fontsize=8.5,color='#333333')
    ax1.set_xlabel('% of categorized low-confidence cases',fontsize=9)
    ax1.set_xlim(0,43)
    ax1.xaxis.grid(True,linestyle='--',alpha=0.4,color='#cccccc')
    ax1.set_axisbelow(True)
    for sp in ['top','right','left']: ax1.spines[sp].set_visible(False)
    ax1.tick_params(axis='y',length=0,labelsize=9)
    ax1.tick_params(axis='x',labelsize=8)
    ax1.set_title('Distribution of categorized cases',fontsize=10,pad=8,fontweight='bold')

    x_min = min(conf_vals)-0.006
    ax2.scatter(conf_vals,range(len(CATEGORY_ORDER)),color=CAT_COLORS,s=130,zorder=5)
    for i,c in enumerate(conf_vals):
        ax2.plot([x_min,c],[i,i],color='#cccccc',linewidth=1.5,zorder=1)
        ax2.text(c+0.0008,i,f'{c:.3f}',va='center',fontsize=8.5,color='#333333')
    ax2.axvline(x=weighted_mean,color='#888888',linestyle=':',linewidth=1.2,alpha=0.8,
                label=f'Mean: {weighted_mean:.3f}')
    ax2.set_xlabel('Mean confidence score',fontsize=9)
    ax2.set_xlim(x_min,max(conf_vals)+0.012)
    ax2.set_yticks([])
    ax2.xaxis.grid(True,linestyle='--',alpha=0.4,color='#cccccc')
    ax2.set_axisbelow(True)
    for sp in ['top','right','left']: ax2.spines[sp].set_visible(False)
    ax2.tick_params(axis='x',labelsize=8)
    ax2.set_title('Mean confidence per pattern',fontsize=10,pad=8,fontweight='bold')
    ax2.legend(fontsize=8,frameon=True,framealpha=0.9,edgecolor='#cccccc',loc='lower right')

    plt.tight_layout(w_pad=2)
    plt.savefig('figures/figure3_combined.png',dpi=180,bbox_inches='tight',facecolor='white')
    plt.show()
    print('Figure 3 saved.')
else:
    print(f'File not found: {LC_FILE}')
    print('Download from https://doi.org/10.5281/zenodo.15085522 and upload to this session.')

---
## 6. Download Results

Download all generated files to your computer.

In [ ]:
from google.colab import files

# Download results CSV
files.download('sentiment_classification_results.csv')

# Download figures
for fig_file in ['figures/figure1_donut.png',
                 'figures/figure2_stacked_bar.png',
                 'figures/figure3_combined.png']:
    if os.path.exists(fig_file):
        files.download(fig_file)

---
## Citation

If you use this code or dataset, please cite:

```bibtex
@article{serna2026crosslingual,
  title   = {Cross-Lingual Sentiment Classification in Sustainable Mobility:
             A Zero-Shot Domain Transfer Evaluation Framework},
  author  = {Serna, Ainhoa and Gerrikagoitia, Jon Kepa and de O{\~n}a, Juan},
  journal = {AI},
  year    = {2026},
  doi     = {10.5281/zenodo.15085522}
}
```

**License:** [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/)